# Digital Image Processing — Filters

This notebook is part of the Digital Image Processing project.

## Part A — RGB Image Processing and Restoration


# Digital Image Processing – Combined Project

This project contains three independent modules:

1. **Huffman Coding** – lossless text compression with encoding and decoding.
2. **Arithmetic Coding** – interval-based lossless text compression with encoding and decoding.
3. **Image Filtering and Restoration** – RGB image input, RGB-to-HSI conversion, filtering of the **Intensity (I) component only**, and HSI-to-RGB conversion.

## HSI processing rule
The input image is converted from RGB to HSI. Each filter is applied only to the Intensity component. Hue (H) and Saturation (S) are kept unchanged. The filtered HSI image is converted back to RGB for display.

This avoids directly modifying colour information during intensity enhancement. Some edge-map outputs are normalized for visualization; restoration outputs are clipped to the valid intensity range.

## Requirements
```python
!pip install opencv-python scipy matplotlib numpy
```


## 1. Import libraries and upload an RGB image

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy import ndimage
from google.colab import files

uploaded = files.upload()
image_name = next(iter(uploaded))
bgr_image = cv2.imread(image_name)

if bgr_image is None:
    raise ValueError("The uploaded file could not be read as an image.")

rgb_image = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB)
rgb_image = rgb_image.astype(np.float32) / 255.0

plt.figure(figsize=(6, 5))
plt.imshow(rgb_image)
plt.title("Original RGB Image")
plt.axis("off")
plt.show()


## 2. RGB to HSI conversion
Hue is represented in radians, while Saturation and Intensity are in the range 0–1.

In [ ]:
def rgb_to_hsi(rgb):
    rgb = np.clip(rgb.astype(np.float32), 0, 1)
    R, G, B = rgb[..., 0], rgb[..., 1], rgb[..., 2]

    intensity = (R + G + B) / 3.0
    minimum = np.minimum(np.minimum(R, G), B)
    saturation = np.zeros_like(intensity)
    nonzero_intensity = intensity > 1e-8
    saturation[nonzero_intensity] = 1 - minimum[nonzero_intensity] / intensity[nonzero_intensity]

    numerator = 0.5 * ((R - G) + (R - B))
    denominator = np.sqrt((R - G)**2 + (R - B) * (G - B))
    ratio = np.zeros_like(intensity)
    valid = denominator > 1e-8
    ratio[valid] = np.clip(numerator[valid] / denominator[valid], -1, 1)
    theta = np.arccos(ratio)

    hue = np.where(B <= G, theta, 2 * np.pi - theta)
    hue[~valid] = 0

    return np.stack((hue, saturation, intensity), axis=-1)


def hsi_to_rgb(hsi):
    H = np.mod(hsi[..., 0], 2 * np.pi)
    S = np.clip(hsi[..., 1], 0, 1)
    I = np.clip(hsi[..., 2], 0, 1)

    R = np.zeros_like(I)
    G = np.zeros_like(I)
    B = np.zeros_like(I)

    sector1 = H < 2 * np.pi / 3
    sector2 = (H >= 2 * np.pi / 3) & (H < 4 * np.pi / 3)
    sector3 = H >= 4 * np.pi / 3

    # Sector 1: 0 <= H < 120 degrees
    h = H[sector1]
    B[sector1] = I[sector1] * (1 - S[sector1])
    R[sector1] = I[sector1] * (1 + (S[sector1] * np.cos(h)) /
                                np.maximum(np.cos(np.pi/3 - h), 1e-8))
    G[sector1] = 3 * I[sector1] - (R[sector1] + B[sector1])

    # Sector 2: 120 <= H < 240 degrees
    h = H[sector2] - 2 * np.pi / 3
    R[sector2] = I[sector2] * (1 - S[sector2])
    G[sector2] = I[sector2] * (1 + (S[sector2] * np.cos(h)) /
                                np.maximum(np.cos(np.pi/3 - h), 1e-8))
    B[sector2] = 3 * I[sector2] - (R[sector2] + G[sector2])

    # Sector 3: 240 <= H < 360 degrees
    h = H[sector3] - 4 * np.pi / 3
    G[sector3] = I[sector3] * (1 - S[sector3])
    B[sector3] = I[sector3] * (1 + (S[sector3] * np.cos(h)) /
                                np.maximum(np.cos(np.pi/3 - h), 1e-8))
    R[sector3] = 3 * I[sector3] - (G[sector3] + B[sector3])

    return np.clip(np.stack((R, G, B), axis=-1), 0, 1)


hsi_image = rgb_to_hsi(rgb_image)
H_original = hsi_image[..., 0].copy()
S_original = hsi_image[..., 1].copy()
I_original = hsi_image[..., 2].copy()

reconstructed_rgb = hsi_to_rgb(hsi_image)
print("Maximum RGB reconstruction difference:",
      float(np.max(np.abs(rgb_image - reconstructed_rgb))))

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1); plt.imshow(rgb_image); plt.title("Original RGB"); plt.axis("off")
plt.subplot(1, 3, 2); plt.imshow(H_original, cmap="hsv"); plt.title("Hue (H)"); plt.axis("off")
plt.subplot(1, 3, 3); plt.imshow(I_original, cmap="gray"); plt.title("Intensity (I)"); plt.axis("off")
plt.show()


## 3. Helper functions

The functions below apply a filter to the complete Intensity image. The original Hue and Saturation arrays are retained for every result.

In [ ]:
def make_rgb_from_intensity(filtered_intensity):
    result_hsi = np.stack((H_original, S_original,
                           np.clip(filtered_intensity, 0, 1)), axis=-1)
    return hsi_to_rgb(result_hsi)


def show_result(title, filtered_intensity, cmap="gray"):
    filtered_intensity = np.asarray(filtered_intensity, dtype=np.float32)
    result_rgb = make_rgb_from_intensity(filtered_intensity)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].imshow(rgb_image); axes[0].set_title("Original RGB")
    axes[1].imshow(filtered_intensity, cmap=cmap, vmin=0, vmax=1); axes[1].set_title(title + " (I)")
    axes[2].imshow(result_rgb); axes[2].set_title(title + " (RGB reconstruction)")
    for ax in axes: ax.axis("off")
    plt.tight_layout()
    plt.show()
    return result_rgb


def normalize_for_display(image):
    image = np.asarray(image, dtype=np.float32)
    lo, hi = np.min(image), np.max(image)
    if hi - lo < 1e-10:
        return np.zeros_like(image)
    return (image - lo) / (hi - lo)


def apply_spatial_kernel(image, kernel):
    return ndimage.convolve(image, kernel.astype(np.float32), mode="reflect")


## 4. Spatial-domain smoothing (low-pass) filters

In [ ]:
# 4.1 Mean / arithmetic mean filter
mean_kernel = np.ones((3, 3), dtype=np.float32) / 9
mean_I = apply_spatial_kernel(I_original, mean_kernel)
show_result("Mean Filter", mean_I)

# 4.2 Weighted average filter
weighted_kernel = np.array([[1, 2, 1],
                            [2, 4, 2],
                            [1, 2, 1]], dtype=np.float32) / 16
weighted_I = apply_spatial_kernel(I_original, weighted_kernel)
show_result("Weighted Average Filter", weighted_I)

# 4.3 Gaussian filter
gaussian_I = ndimage.gaussian_filter(I_original, sigma=1.0, mode="reflect")
show_result("Gaussian Filter", gaussian_I)

# 4.4 Median filter
median_I = ndimage.median_filter(I_original, size=3, mode="reflect")
show_result("Median Filter", median_I)


## 5. Additional spatial restoration / order-statistics filters

In [ ]:
# These neighbourhood filters are applied to Intensity only.
def local_geometric_mean(image, size=3, eps=1e-8):
    log_image = np.log(np.maximum(image, eps))
    return np.exp(ndimage.uniform_filter(log_image, size=size, mode="reflect"))

def local_harmonic_mean(image, size=3, eps=1e-8):
    reciprocal = 1.0 / np.maximum(image, eps)
    return 1.0 / ndimage.uniform_filter(reciprocal, size=size, mode="reflect")

def local_contraharmonic_mean(image, Q=1.5, size=3, eps=1e-8):
    numerator = ndimage.uniform_filter(np.maximum(image, 0)**(Q + 1), size=size, mode="reflect")
    denominator = ndimage.uniform_filter(np.maximum(image, 0)**Q, size=size, mode="reflect")
    return numerator / np.maximum(denominator, eps)

def local_min(image, size=3):
    return ndimage.minimum_filter(image, size=size, mode="reflect")

def local_max(image, size=3):
    return ndimage.maximum_filter(image, size=size, mode="reflect")

def local_midpoint(image, size=3):
    return (local_min(image, size) + local_max(image, size)) / 2

def alpha_trimmed_mean(image, size=3, d=2):
    if d < 0 or d >= size * size or d % 2:
        raise ValueError("d must be an even number from 0 to size*size-1.")
    def trimmed(values):
        values = np.sort(values)
        if d:
            values = values[d//2:len(values)-d//2]
        return np.mean(values)
    return ndimage.generic_filter(image, trimmed, size=size, mode="reflect")

geometric_I = local_geometric_mean(I_original)
show_result("Geometric Mean Filter", geometric_I)

harmonic_I = local_harmonic_mean(I_original)
show_result("Harmonic Mean Filter", harmonic_I)

# Q is positive here; Q sign should be selected according to the noise type.
contraharmonic_I = local_contraharmonic_mean(I_original, Q=1.5)
show_result("Contraharmonic Mean Filter (Q=1.5)", contraharmonic_I)

minimum_I = local_min(I_original)
show_result("Minimum Filter", minimum_I)

maximum_I = local_max(I_original)
show_result("Maximum Filter", maximum_I)

midpoint_I = local_midpoint(I_original)
show_result("Midpoint Filter", midpoint_I)

alpha_trimmed_I = alpha_trimmed_mean(I_original, size=3, d=2)
show_result("Alpha-Trimmed Mean Filter (d=2)", alpha_trimmed_I)


## 6. Spatial-domain sharpening filters

In [ ]:
# 6.1 Laplacian sharpening
laplacian_kernel = np.array([[0, 1, 0],
                             [1, -4, 1],
                             [0, 1, 0]], dtype=np.float32)
laplacian_response = apply_spatial_kernel(I_original, laplacian_kernel)
laplacian_sharp_I = np.clip(I_original - laplacian_response, 0, 1)
show_result("Laplacian Sharpening", laplacian_sharp_I)

# 6.2 Sobel edge detection
sobel_x_kernel = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
sobel_y_kernel = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float32)
Gx = apply_spatial_kernel(I_original, sobel_x_kernel)
Gy = apply_spatial_kernel(I_original, sobel_y_kernel)
sobel_magnitude = np.sqrt(Gx**2 + Gy**2)
sobel_display = normalize_for_display(sobel_magnitude)
show_result("Sobel Edge Map (normalized)", sobel_display)

# 6.3 High-boost filtering
A = 1.5
blurred_for_boost = apply_spatial_kernel(I_original, mean_kernel)
high_frequency_mask = I_original - blurred_for_boost
high_boost_I = np.clip(A * I_original - blurred_for_boost, 0, 1)
show_result("High-Boost Filter (A=1.5)", high_boost_I)

# 6.3 Prewitt edge detection
prewitt_x_kernel = np.array([[-1, 0, 1],
                             [-1, 0, 1],
                             [-1, 0, 1]], dtype=np.float32)
prewitt_y_kernel = np.array([[-1, -1, -1],
                             [ 0,  0,  0],
                             [ 1,  1,  1]], dtype=np.float32)
prewitt_Gx = apply_spatial_kernel(I_original, prewitt_x_kernel)
prewitt_Gy = apply_spatial_kernel(I_original, prewitt_y_kernel)
prewitt_magnitude = np.sqrt(prewitt_Gx**2 + prewitt_Gy**2)
prewitt_display = normalize_for_display(prewitt_magnitude)
show_result("Prewitt Edge Map (normalized)", prewitt_display)


## 7. Frequency-domain filters

The Fourier transform is applied to the complete Intensity image. A centred frequency mask is multiplied by the shifted spectrum, then the inverse Fourier transform produces the filtered Intensity image.

In [ ]:
def frequency_filter(image, mask):
    F = np.fft.fftshift(np.fft.fft2(image))
    G = F * mask
    result = np.fft.ifft2(np.fft.ifftshift(G)).real
    return result

def frequency_distance(shape):
    rows, cols = shape
    u = np.arange(rows) - rows // 2
    v = np.arange(cols) - cols // 2
    U, V = np.meshgrid(v, u)
    return np.sqrt(U**2 + V**2)

D = frequency_distance(I_original.shape)
D0 = max(1.0, min(I_original.shape) * 0.12)
n = 2

ideal_lpf_mask = (D <= D0).astype(np.float32)
butterworth_lpf_mask = 1 / (1 + (D / D0)**(2*n))
gaussian_lpf_mask = np.exp(-(D**2) / (2 * D0**2))

ideal_hpf_mask = 1 - ideal_lpf_mask
butterworth_hpf_mask = 1 - butterworth_lpf_mask
gaussian_hpf_mask = 1 - gaussian_lpf_mask

frequency_results = {
    "Ideal Low-Pass Filter": frequency_filter(I_original, ideal_lpf_mask),
    "Butterworth Low-Pass Filter": frequency_filter(I_original, butterworth_lpf_mask),
    "Gaussian Low-Pass Filter": frequency_filter(I_original, gaussian_lpf_mask),
    "Ideal High-Pass Filter": frequency_filter(I_original, ideal_hpf_mask),
    "Butterworth High-Pass Filter": frequency_filter(I_original, butterworth_hpf_mask),
    "Gaussian High-Pass Filter": frequency_filter(I_original, gaussian_hpf_mask),
}

for name, result in frequency_results.items():
    # High-pass results are edge/detail maps; normalize only for display.
    display_result = normalize_for_display(result) if "High-Pass" in name else np.clip(result, 0, 1)
    show_result(name, display_result)


## 8. Band and notch frequency filters
Notch centres are specified as offsets from the centre of the shifted Fourier spectrum. Adjust the offsets after inspecting the spectrum of a periodic-noise image.

In [ ]:
# Band reject and band pass masks
band_center = D0
band_width = max(2.0, min(I_original.shape) * 0.025)
band_reject_mask = ~((D >= band_center - band_width/2) &
                     (D <= band_center + band_width/2))
band_reject_mask = band_reject_mask.astype(np.float32)
band_pass_mask = 1 - band_reject_mask

band_reject_I = frequency_filter(I_original, band_reject_mask)
band_pass_I = frequency_filter(I_original, band_pass_mask)
show_result("Band Reject Filter", np.clip(band_reject_I, 0, 1))
show_result("Band Pass Filter (normalized)", normalize_for_display(band_pass_I))

# Notch reject/pass: symmetric notch pair at configurable offsets.
def notch_masks(shape, offsets=((30, 30),), radius=8):
    rows, cols = shape
    Y, X = np.ogrid[:rows, :cols]
    cy, cx = rows // 2, cols // 2
    reject = np.ones(shape, dtype=np.float32)
    for dy, dx in offsets:
        d1 = (Y - (cy + dy))**2 + (X - (cx + dx))**2
        d2 = (Y - (cy - dy))**2 + (X - (cx - dx))**2
        reject[(d1 <= radius**2) | (d2 <= radius**2)] = 0
    return reject, 1 - reject

notch_reject_mask, notch_pass_mask = notch_masks(I_original.shape,
                                                  offsets=((30, 30),), radius=8)
notch_reject_I = frequency_filter(I_original, notch_reject_mask)
notch_pass_I = frequency_filter(I_original, notch_pass_mask)
show_result("Notch Reject Filter", np.clip(notch_reject_I, 0, 1))
show_result("Notch Pass Filter (normalized)", normalize_for_display(notch_pass_I))


## 9. Image restoration filters

The restoration examples below create a known degradation from the original Intensity image, then attempt restoration. A Gaussian point-spread function (PSF) models blur. The same PSF is used by inverse, Wiener and constrained least-squares restoration.

In [ ]:
def psf_to_otf(psf, shape):
    padded = np.zeros(shape, dtype=np.float32)
    ph, pw = psf.shape
    padded[:ph, :pw] = psf
    padded = np.roll(padded, -(ph // 2), axis=0)
    padded = np.roll(padded, -(pw // 2), axis=1)
    return np.fft.fft2(padded)

def gaussian_psf(size=9, sigma=1.5):
    ax = np.arange(-(size//2), size//2 + 1)
    X, Y = np.meshgrid(ax, ax)
    psf = np.exp(-(X**2 + Y**2) / (2*sigma**2))
    return (psf / psf.sum()).astype(np.float32)

psf = gaussian_psf(size=9, sigma=1.5)
H_otf = psf_to_otf(psf, I_original.shape)
F_original = np.fft.fft2(I_original)

# Simulate blurred and noisy degradation
blurred = np.fft.ifft2(F_original * H_otf).real
rng = np.random.default_rng(7)
noise_sigma = 0.01
degraded = np.clip(blurred + rng.normal(0, noise_sigma, blurred.shape), 0, 1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1); plt.imshow(I_original, cmap="gray"); plt.title("Original Intensity"); plt.axis("off")
plt.subplot(1, 3, 2); plt.imshow(blurred, cmap="gray"); plt.title("Blurred"); plt.axis("off")
plt.subplot(1, 3, 3); plt.imshow(degraded, cmap="gray"); plt.title("Blurred + Gaussian Noise"); plt.axis("off")
plt.show()


In [ ]:
# 9.1 Inverse filtering (regularized denominator avoids division by zero)
G_degraded = np.fft.fft2(degraded)
inverse_restored = np.fft.ifft2(
    G_degraded / np.where(np.abs(H_otf) < 1e-6, 1e-6, H_otf)
).real
inverse_restored = np.clip(inverse_restored, 0, 1)
show_result("Inverse Filtering Restoration", inverse_restored)

# 9.2 Wiener filtering
K = 0.001  # approximate noise-to-signal power ratio
H_conjugate = np.conjugate(H_otf)
wiener_restored = np.fft.ifft2(
    (H_conjugate / (np.abs(H_otf)**2 + K)) * G_degraded
).real
wiener_restored = np.clip(wiener_restored, 0, 1)
show_result("Wiener Filtering Restoration", wiener_restored)

# 9.3 Constrained Least Squares (CLS) filtering
laplacian_psf = np.array([[0, -1, 0],
                          [-1, 4, -1],
                          [0, -1, 0]], dtype=np.float32)
P_otf = psf_to_otf(laplacian_psf, I_original.shape)
gamma = 0.001
cls_restored = np.fft.ifft2(
    (H_conjugate / (np.abs(H_otf)**2 + gamma*np.abs(P_otf)**2 + 1e-8))
    * G_degraded
).real
cls_restored = np.clip(cls_restored, 0, 1)
show_result("Constrained Least Squares Restoration", cls_restored)


## 10. Noise-specific restoration demonstrations

- Mean filtering is demonstrated on additive Gaussian noise.
- Median filtering is demonstrated on salt-and-pepper noise.
- Band/notch rejection is intended for periodic noise; the notch locations should be chosen from peaks in the shifted Fourier magnitude spectrum.

In [ ]:
# Gaussian noise + mean filter
gaussian_noisy = np.clip(I_original + rng.normal(0, 0.05, I_original.shape), 0, 1)
gaussian_denoised = ndimage.uniform_filter(gaussian_noisy, size=3, mode="reflect")

# Salt-and-pepper noise + median filter
sp_noisy = I_original.copy()
salt_pepper = rng.random(I_original.shape)
sp_noisy[salt_pepper < 0.025] = 0
sp_noisy[salt_pepper > 0.975] = 1
sp_denoised = ndimage.median_filter(sp_noisy, size=3, mode="reflect")

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
items = [
    (I_original, "Original Intensity"),
    (gaussian_noisy, "Gaussian Noisy"),
    (gaussian_denoised, "Mean Filter Result"),
    (I_original, "Original Intensity"),
    (sp_noisy, "Salt-and-Pepper Noisy"),
    (sp_denoised, "Median Filter Result"),
]
for ax, (img, title) in zip(axes.flat, items):
    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

show_result("Gaussian Noise Removed by Mean Filter", gaussian_denoised)
show_result("Salt-and-Pepper Noise Removed by Median Filter", sp_denoised)


## 11. Export filtered images

Choose any result variable created above and save its RGB reconstruction.

In [ ]:
# Example: save the Wiener-restored RGB image.
final_rgb = make_rgb_from_intensity(wiener_restored)
final_uint8 = (np.clip(final_rgb, 0, 1) * 255).astype(np.uint8)
cv2.imwrite("wiener_restored_rgb.png",
            cv2.cvtColor(final_uint8, cv2.COLOR_RGB2BGR))
print("Saved: wiener_restored_rgb.png")
